# Project 2: Does Superstar Consistency or Volatility Drive Playoff Success?
### A Machine Learning Investigation Using Advanced NBA Metrics

---

**Research Question:**
> How does the variance of a superstar's offensive impact metric — combining points created, empty possessions, usage rate, and per-75 possession normalization — relate to team playoff winning percentage?

**Core Question:**
> Does *consistency* or *volatility* in superstar offensive performance matter more for winning in the playoffs, and can we quantify that relationship using machine learning?

---

## Project Overview

This project investigates whether it matters *how consistently* a superstar performs in the playoffs, or whether their overall talent level simply overwhelms game-to-game variation as a predictor of team success.

### Data Sources
| Source | What It Provides |
|---|---|
| `PlayerStatistics.csv` | Game-by-game box scores for every NBA player, 1985–present |
| Basketball Reference Advanced Stats | Season-level: VORP, BPM, WS, NetRtg, PER (scraped via `bbref_scraper.py`) |

### How the Dataset Was Built
1. **Superstar identification:** Players averaging ≥28 PPG over ≥50 regular season games qualify as "superstars" for that season
2. **Per-game metrics computed:** For each playoff game log, we calculate TS%, eFG%, TOV%, and OIS/75
3. **Variability features:** For each (player, season), we summarize the *distribution* of those per-game metrics — mean, std, CV, IQR, floor (p10), ceiling (p90)
4. **BBRef merge:** Season-level VORP, BPM, Win Shares, Net Rating merged in by player + season
5. **Target variable:** Team playoff win % for that player's team in that season

### Final Dataset
- **109 player-seasons**, 45 unique superstars
- **42 features** across variability and quality categories
- Covers 1985–2026 playoffs

## Section 1: Setup & Imports

All libraries used in this project. Key packages:
- `pandas` / `numpy` — data manipulation
- `sklearn` — all machine learning models and evaluation tools
- `scipy.stats` — statistical correlation tests
- `matplotlib` / `seaborn` — visualizations

In [ ]:
# ── Standard data libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# ── Visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Statistical tests (Pearson r, Spearman rho)
from scipy import stats
from scipy.stats import pearsonr, spearmanr

# ── Machine learning models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# ── ML utilities: pipelines, scaling, cross-validation, evaluation
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import cross_validate, KFold, GridSearchCV
from sklearn.metrics import r2_score
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer  # handles missing BBRef values

# ── Reproducibility: fix random seed so results are identical each run
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries loaded successfully.")

## Section 2: Load & Inspect the Dataset

The modeling dataset was pre-built by two scripts:
- `bbref_scraper.py` — scraped Basketball Reference for VORP, BPM, WS, NetRtg
- `bbref_merge_and_features.py` — merged BBRef stats with per-game variability features

The result is one row per (player, season) in the playoffs.

In [ ]:
# ── Load the pre-built modeling dataset
# Each row = one superstar's playoff performance in one season
df = pd.read_csv("modeling_dataset_final.csv")

# ── Basic dataset overview
print(f"Dataset shape:     {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Unique superstars: {df['playerName'].nunique()}")
print(f"Seasons covered:   {df['season_label'].nunique()}")

# ── Target variable distribution
# playoff_win_pct ranges from 0 (lost every game) to 1 (won every game)
# 0.5 = exactly .500 in playoffs, 1.0 = perfect postseason (rare)
print(f"\nTarget Variable — Playoff Win %:")
print(df['playoff_win_pct'].describe().round(3))

# ── Check how much BBRef data we have (not all seasons had full coverage)
print(f"\nBBRef metric coverage (% of rows with data):")
for col in ['VORP','BPM','WS','WS/48','NetRtg','PER']:
    pct = df[col].notna().mean()*100
    print(f"  {col:<10} {pct:.1f}%")

# ── Preview the key columns
print("\nSample rows:")
df[['playerName','season_label','playoff_win_pct',
    'VORP','BPM','WS','NetRtg',
    'mean_TS_pct','std_TS_pct','cv_OIS_per75','playoff_games']].head(10)

## Section 3: Feature Architecture

We separate features into **three tiers** and run separate ML experiments for each. This is intentional — it lets us directly answer:
- *Can variability alone predict playoff success?*
- *Does raw talent (VORP/BPM/WS) predict better than variability?*
- *Does combining both give the best result?*

### Understanding the Metrics

**Variability Features (computed from per-game playoff logs):**
- `mean_X` — the player's average level of metric X across playoff games
- `std_X` — standard deviation; how much it fluctuates game to game
- `cv_X` — coefficient of variation (std ÷ mean); normalized volatility so we can compare players at different performance levels
- `iqr_X` — interquartile range; robust spread measure that ignores extreme outliers
- `p10_X` — 10th percentile; the player's "floor" (worst typical games)
- `p90_X` — 90th percentile; the player's "ceiling" (best typical games)

**BBRef Season Metrics (full-season averages from Basketball Reference):**
- `VORP` — Value Over Replacement Player; how many points per 100 possessions better than a replacement-level player
- `BPM` — Box Plus/Minus; overall on/off impact estimated from box score
- `WS/48` — Win Shares per 48 minutes; efficiency of contribution to team wins
- `NetRtg` — Net Rating; points scored minus allowed per 100 possessions with player on floor
- `PER` — Player Efficiency Rating; per-minute production summary

In [ ]:
# ── TIER 1: Variability features — computed from per-game playoff logs
# These capture HOW a player performs: are they consistent or volatile?
# 100% data coverage — no missing values
VARIABILITY_FEATURES = [
    # True Shooting % variability (shooting efficiency including FTs and 3s)
    "mean_TS_pct", "std_TS_pct", "cv_TS_pct", "iqr_TS_pct",
    "p10_TS_pct",  "p90_TS_pct",

    # Effective FG% variability (weights 3-pointers properly)
    "mean_eFG_pct", "std_eFG_pct", "cv_eFG_pct", "iqr_eFG_pct",
    "p10_eFG_pct",  "p90_eFG_pct",

    # Offensive Impact Score per 75 possessions variability
    # OIS = (points created - empty possessions) / possessions used × 75
    "mean_OIS_per75", "std_OIS_per75", "cv_OIS_per75", "iqr_OIS_per75",
    "p10_OIS_per75",  "p90_OIS_per75",

    # Points per possession variability (pure scoring efficiency)
    "mean_pts_per_poss", "std_pts_per_poss", "iqr_pts_per_poss",
    "p10_pts_per_poss",  "p90_pts_per_poss",

    # Turnover rate variability (lower TOV% = better)
    "mean_TOV_pct", "p10_TOV_pct",

    # Game count and elite performance frequency
    "playoff_games",   # total playoff games played this season
    "pct_elite_TS",    # % of games where TS% exceeded player's own 75th percentile
    "pct_elite_OIS",   # % of games where OIS exceeded player's own 75th percentile
]

# ── TIER 2: BBRef season-level quality metrics
# These capture HOW GOOD a player is overall — their talent baseline
# ~75% data coverage (some older seasons missing)
BBREF_FEATURES = [
    "VORP",    # Value Over Replacement Player
    "BPM",     # Box Plus/Minus (offense + defense)
    "OBPM",    # Offensive Box Plus/Minus only
    "DBPM",    # Defensive Box Plus/Minus only
    "PER",     # Player Efficiency Rating
    "WS",      # Total Win Shares
    "WS/48",   # Win Shares per 48 minutes (efficiency version)
    "NetRtg",  # Net Rating (team points differential with player on floor)
]

# ── TIER 3: Everything combined
ALL_FEATURES = VARIABILITY_FEATURES + BBREF_FEATURES

TARGET = "playoff_win_pct"

print(f"Variability features: {len(VARIABILITY_FEATURES)}")
print(f"BBRef features:       {len(BBREF_FEATURES)}")
print(f"Total features:       {len(ALL_FEATURES)}")
print(f"Target variable:      {TARGET}")

## Section 4: Exploratory Analysis — Which Features Correlate with Playoff Win %?

Before running ML models, we first look at simple pairwise correlations. This tells us which features have *linear* relationships with playoff win%.

**Pearson r** measures linear correlation (−1 to +1).
**Spearman ρ** measures rank-order (monotonic) correlation — less sensitive to outliers.

A `*` indicates the result is statistically significant (p < 0.05), meaning we can be reasonably confident the relationship isn't just random chance.

In [ ]:
# ── Compute Pearson and Spearman correlations for every feature vs playoff win%
corr_results = []
for col in ALL_FEATURES:
    sub = df[[col, TARGET]].dropna()
    if len(sub) < 15:  # need at least 15 data points for meaningful correlation
        continue
    r,  p  = pearsonr(sub[col],  sub[TARGET])
    sr, sp = spearmanr(sub[col], sub[TARGET])
    corr_results.append({
        "Feature":    col,
        "Pearson_r":  round(r,  3),
        "Pearson_p":  round(p,  4),
        "Spearman_r": round(sr, 3),
        "Spearman_p": round(sp, 4),
        "Significant": "✓ *" if p < 0.05 else "",
        "Category":   "BBRef Quality" if col in BBREF_FEATURES else "Variability"
    })

# Sort by absolute Pearson r so strongest relationships appear first
corr_df = pd.DataFrame(corr_results).sort_values("Pearson_r", ascending=False, key=abs)

print("Feature Correlations with Playoff Win % (sorted by absolute r):")
print("(✓ = statistically significant at p < 0.05)")
print()
print(corr_df[["Feature","Pearson_r","Pearson_p","Spearman_r","Significant","Category"]].to_string(index=False))

In [ ]:
# ── Visualization: Correlation bar chart
# This chart shows every feature's Pearson r with playoff win%
# Green bars = positive relationship (higher feature → higher win%)
# Red bars = negative relationship (higher feature → lower win%)
# The dotted lines at ±0.3 mark a "moderate" correlation threshold

fig, ax = plt.subplots(figsize=(10, 12))

colors = ["#2ecc71" if r >= 0 else "#e74c3c" for r in corr_df["Pearson_r"]]
ax.barh(corr_df["Feature"], corr_df["Pearson_r"],
        color=colors, edgecolor="white", height=0.7)

# Reference lines
ax.axvline(0,    color="black", lw=1.0)
ax.axvline(0.3,  color="grey",  lw=0.8, ls="--", alpha=0.6, label="Moderate (r=±0.3)")
ax.axvline(-0.3, color="grey",  lw=0.8, ls="--", alpha=0.6)

# Annotate significant features
for _, row in corr_df.iterrows():
    if row["Pearson_p"] < 0.05:
        x_pos = row["Pearson_r"] + (0.01 if row["Pearson_r"] >= 0 else -0.01)
        ax.text(x_pos, list(corr_df["Feature"]).index(row["Feature"]),
                "✓", va="center", fontsize=8, color="black")

ax.set_xlabel("Pearson r with Playoff Win %", fontsize=11)
ax.set_title("All Feature Correlations with Playoff Win %\n"
             "(green = positive, red = negative, ✓ = p<0.05 significant)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# ── KEY FINDING: Print the top correlators
print("\nTop 8 strongest correlators with playoff win%:")
print(corr_df[["Feature","Pearson_r","Pearson_p","Category"]].head(8).to_string(index=False))
print("\nKey takeaway: BBRef quality metrics (WS/48, NetRtg, VORP, BPM) tend to")
print("correlate more strongly than pure variability measures (std, cv).")

## Section 5: Key Scatter Plots

We plot the **8 most important feature relationships** — 4 quality metrics and 4 variability metrics — to visually see how each one relates to playoff win%.

Each plot shows:
- Each dot = one (player, season) observation
- The dashed line = linear trend
- `r=` in the legend = Pearson correlation (and * if significant)

**What to look for:** Steep trend lines and tight clusters around the line = strong relationship. Flat lines or scattered dots = weak or no relationship.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

# ── The 8 features we're highlighting — 4 quality, 4 variability
# Chosen based on having the strongest correlations in each category
plot_pairs = [
    # Quality metrics (BBRef) — these measure HOW GOOD the player is
    ("WS/48",           "Win Shares per 48 min\n(Quality: how efficient overall?)",   "#e74c3c"),
    ("NetRtg",          "Net Rating\n(Quality: team impact on floor?)",               "#e67e22"),
    ("VORP",            "VORP\n(Quality: value over replacement?)",                   "#f39c12"),
    ("PER",             "PER\n(Quality: per-minute efficiency?)",                     "#27ae60"),
    # Variability metrics — these measure HOW CONSISTENT the player is
    ("mean_pts_per_poss","Mean Pts per Possession\n(Avg scoring efficiency)",         "#2980b9"),
    ("p90_OIS_per75",   "OIS Ceiling — p90\n(Best games: how dominant?)",            "#8e44ad"),
    ("cv_OIS_per75",    "CV of OIS per 75\n(Volatility: how unpredictable?)",        "#c0392b"),
    ("std_TS_pct",      "Std Dev of TS%\n(Shooting consistency game-to-game)",       "#16a085"),
]

for i, (feat, label, color) in enumerate(plot_pairs):
    sub = df[[feat, TARGET]].dropna()
    axes[i].scatter(sub[feat], sub[TARGET],
                   color=color, alpha=0.7, s=55,
                   edgecolors="white", linewidths=0.5)

    # Add regression line
    m, b, r, p, _ = stats.linregress(sub[feat], sub[TARGET])
    x_line = np.linspace(sub[feat].min(), sub[feat].max(), 100)
    sig_marker = "*" if p < 0.05 else ""
    axes[i].plot(x_line, m*x_line + b, "k--", lw=1.5,
                label=f"r={r:.2f}{sig_marker}")
    axes[i].set_xlabel(label, fontsize=8)
    axes[i].set_ylabel("Playoff Win %" if i % 4 == 0 else "", fontsize=9)
    axes[i].legend(fontsize=9)

    # Add category label
    cat = "Quality Metric" if i < 4 else "Variability Metric"
    axes[i].set_title(f"[{cat}]", fontsize=8, color="grey")

# Add row labels
fig.text(0.01, 0.75, "QUALITY\nMETRICS", fontsize=10, fontweight="bold",
         color="#c0392b", va="center", rotation=90)
fig.text(0.01, 0.28, "VARIABILITY\nMETRICS", fontsize=10, fontweight="bold",
         color="#2980b9", va="center", rotation=90)

plt.suptitle("Quality Metrics vs. Variability Metrics — Which Predicts Playoff Win % Better?\n"
             "(* = statistically significant at p<0.05)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("INTERPRETATION:")
print("Top row (quality metrics): These show clearer upward trends — better players = more wins.")
print("Bottom row (variability metrics): Flatter trend lines — volatility has weaker predictive power.")
print("The ceiling metric (p90_OIS_per75) is the strongest variability predictor.")

## Section 6: High vs. Low Volatility — Does It Make a Difference?

We split superstars into two groups at the **median CV of OIS** (coefficient of variation — our primary volatility measure). Then we compare their playoff win percentages using:
- **Violin plots** — show the full distribution shape
- **Welch's t-test** — statistical test for whether the group means are significantly different (p < 0.05 = significant)

We run this same analysis for **WS/48** (quality) as a comparison — to show that quality grouping IS significant while volatility grouping may not be.

In [ ]:
# ── Split into high/low volatility groups at the median CV of OIS
# CV = std / mean; higher CV = more volatile/inconsistent performance
median_cv = df["cv_OIS_per75"].median()
df["volatility_group"] = np.where(df["cv_OIS_per75"] >= median_cv,
                                   "High Volatility", "Low Volatility")

# ── Group statistics
vol_stats = (df.groupby("volatility_group")["playoff_win_pct"]
             .agg(["mean","std","count"])
             .rename(columns={"mean":"Avg Win%","std":"Std","count":"N"}))
print("Volatility Groups (split at median CV of OIS):")
print(vol_stats.round(3))

# ── Statistical significance test
hi_vol = df[df["volatility_group"]=="High Volatility"]["playoff_win_pct"]
lo_vol = df[df["volatility_group"]=="Low Volatility"]["playoff_win_pct"]
t1, p1 = stats.ttest_ind(hi_vol, lo_vol, equal_var=False)
print(f"\nWelch t-test (volatility): t={t1:.3f}, p={p1:.4f}")
print("→ " + ("SIGNIFICANT difference." if p1 < 0.05 else "No significant difference between groups."))

# ── Same analysis for WS/48 (quality) for comparison
median_ws = df["WS/48"].median()
df["ws_group"] = np.where(df["WS/48"] >= median_ws, "High WS/48", "Low WS/48")
hi_ws = df[df["ws_group"]=="High WS/48"]["playoff_win_pct"].dropna()
lo_ws = df[df["ws_group"]=="Low WS/48"]["playoff_win_pct"].dropna()
t2, p2 = stats.ttest_ind(hi_ws, lo_ws, equal_var=False)

ws_stats = (df.dropna(subset=["WS/48"]).groupby("ws_group")["playoff_win_pct"]
            .agg(["mean","std","count"])
            .rename(columns={"mean":"Avg Win%","std":"Std","count":"N"}))
print(f"\nWin Shares/48 Groups (quality measure):")
print(ws_stats.round(3))
print(f"Welch t-test (WS/48): t={t2:.3f}, p={p2:.4f}")
print("→ " + ("SIGNIFICANT difference." if p2 < 0.05 else "No significant difference."))

# ── VISUALIZATION: Side by side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Volatility groups
sns.violinplot(data=df, x="volatility_group", y="playoff_win_pct",
               palette=["#e74c3c","#2ecc71"], inner="box", ax=axes[0])
axes[0].set_title(f"Playoff Win% by OIS Volatility\n"
                  f"(t={t1:.2f}, p={p1:.3f} — {'significant' if p1<0.05 else 'NOT significant'})",
                  fontsize=11, fontweight="bold")
axes[0].set_xlabel("Volatility Group (split at median CV)", fontsize=10)
axes[0].set_ylabel("Playoff Win %", fontsize=10)
# The violin shape shows distribution — wide = many players at that win%
# The box inside shows median (center line) and IQR (box width)

# Right: Quality groups
sns.violinplot(data=df.dropna(subset=["WS/48"]), x="ws_group", y="playoff_win_pct",
               palette=["#3498db","#e67e22"], inner="box", ax=axes[1])
axes[1].set_title(f"Playoff Win% by Win Shares/48 (Quality)\n"
                  f"(t={t2:.2f}, p={p2:.3f} — {'significant' if p2<0.05 else 'NOT significant'})",
                  fontsize=11, fontweight="bold")
axes[1].set_xlabel("WS/48 Group (split at median)", fontsize=10)
axes[1].set_ylabel("", fontsize=10)

plt.suptitle("Does Splitting Superstars by Volatility vs. Quality Separate Playoff Winners?",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\nINTERPRETATION:")
print("Compare the p-values above. If quality (WS/48) splits are significant")
print("but volatility splits are not, that confirms quality > consistency as a predictor.")

## Section 7: Machine Learning Models — Three Experiments

We run **6 regression models** across **3 separate experiments**:

| Experiment | Features Used | Purpose |
|---|---|---|
| A | Variability only (100% coverage) | Can consistency/volatility alone predict win%? |
| B | BBRef metrics only (75% coverage) | Can raw talent alone predict win%? |
| C | All features combined | Does combining both improve predictions? |

### Why These 6 Models?
- **Linear Regression** — baseline; assumes simple straight-line relationships
- **Ridge** — adds L2 penalty to prevent overfitting; handles correlated features better than plain linear
- **Lasso** — adds L1 penalty that can zero out unimportant features (automatic feature selection)
- **ElasticNet** — combines Ridge + Lasso penalties; often best on mixed feature sets
- **Random Forest** — hundreds of decision trees averaged together; captures non-linear patterns
- **Gradient Boosting** — trees built sequentially, each correcting previous errors; often strongest on tabular data

### Evaluation: 5-Fold Cross-Validation
The data is split into 5 chunks. Each chunk is used once as a test set, 4 times as training. Final score = average across all 5 folds. This prevents overfitting and gives a realistic estimate of how the model would perform on new data.

**CV R²** = key metric. 0 = model explains nothing, 1 = perfect predictions. Negative = worse than just predicting the mean.
**CV RMSE** = average prediction error in win% units (e.g., 0.15 = off by 15 percentage points on average).
**Overfit Gap** = Train R² minus CV R². Large gap means the model memorized training data but doesn't generalize.

In [ ]:
# ── Set up 5-fold cross-validation
# shuffle=True randomizes which games go in which fold
# random_state=42 makes results reproducible
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def run_experiment(X_raw, y_raw, label):
    """
    Run all 6 ML models on a given feature set.
    Each model is wrapped in a Pipeline that:
      1. Imputes missing values (median strategy)
      2. Scales features (StandardScaler for linear models, RobustScaler for tree models)
      3. Fits the model
    Returns a DataFrame of results plus the aligned X and y.
    """
    # Align features and target, drop rows where target is missing
    data = pd.concat([X_raw, y_raw], axis=1).dropna(subset=[TARGET])
    X = data[X_raw.columns]
    y = data[TARGET]

    print(f"\n{'='*55}")
    print(f"EXPERIMENT: {label}")
    print(f"  Samples: {len(X)}, Features: {X.shape[1]}")
    print(f"  Target range: {y.min():.3f} to {y.max():.3f}")
    print(f"{'='*55}")

    models = {
        # Linear Regression: baseline, no regularization
        "Linear Regression":
            Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler()),
                      ("mdl", LinearRegression())]),

        # Ridge: L2 regularization — shrinks all coefficients toward zero
        # Good for correlated features (std and CV are correlated with each other)
        "Ridge":
            Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler()),
                      ("mdl", Ridge(alpha=1.0))]),

        # Lasso: L1 regularization — drives some coefficients to exactly zero
        # Effectively performs automatic feature selection
        "Lasso":
            Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler()),
                      ("mdl", Lasso(alpha=0.01, max_iter=10000))]),

        # ElasticNet: combines L1 and L2 — flexible mix of Ridge and Lasso
        "ElasticNet":
            Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler()),
                      ("mdl", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000))]),

        # Random Forest: 300 decision trees, each trained on random subsets
        # RobustScaler used — less sensitive to outlier games in the dataset
        "Random Forest":
            Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", RobustScaler()),
                      ("mdl", RandomForestRegressor(n_estimators=300, max_depth=4,
                                                    min_samples_leaf=3,
                                                    random_state=RANDOM_STATE))]),

        # Gradient Boosting: sequential trees, each correcting previous errors
        # subsample=0.8 adds randomness to prevent overfitting
        "Gradient Boosting":
            Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", RobustScaler()),
                      ("mdl", GradientBoostingRegressor(n_estimators=200,
                                                        learning_rate=0.05,
                                                        max_depth=3, subsample=0.8,
                                                        random_state=RANDOM_STATE))]),
    }

    results = []
    print(f"  {'Model':<22} {'CV R²':>8} {'CV RMSE':>9} {'Train R²':>10} {'Overfit':>8}")
    print(f"  {'-'*60}")
    for name, pipe in models.items():
        cv_scores = cross_validate(pipe, X, y, cv=kf,
                                   scoring=["r2","neg_root_mean_squared_error"],
                                   return_train_score=True)
        r2   = cv_scores["test_r2"].mean()
        rmse = -cv_scores["test_neg_root_mean_squared_error"].mean()
        tr2  = cv_scores["train_r2"].mean()
        overfit = tr2 - r2
        results.append({"Model": name, "Experiment": label,
                        "CV_R2": round(r2,4), "CV_RMSE": round(rmse,4),
                        "Train_R2": round(tr2,4), "Overfit": round(overfit,4)})
        # Flag models with large overfit gaps
        overfit_flag = " ⚠ overfit" if overfit > 0.2 else ""
        print(f"  {name:<22} {r2:>8.4f} {rmse:>9.4f} {tr2:>10.4f} {overfit:>8.4f}{overfit_flag}")

    return pd.DataFrame(results), X, y

# ── Run all three experiments
res_A, X_A, y_A = run_experiment(df[VARIABILITY_FEATURES], df[TARGET],
                                  "A: Variability Only")
res_B, X_B, y_B = run_experiment(df[BBREF_FEATURES],       df[TARGET],
                                  "B: BBRef Quality Only")
res_C, X_C, y_C = run_experiment(df[ALL_FEATURES],         df[TARGET],
                                  "C: All Features Combined")

all_results = pd.concat([res_A, res_B, res_C], ignore_index=True)

## Section 8: Model Comparison — Which Experiment Wins?

This section directly answers: **does variability, quality, or their combination best predict playoff success?**

The grouped bar chart shows CV R² for each model across the three experiments. The best-per-experiment bar shows which approach dominates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Left chart: grouped bars by experiment
# Each cluster = one model; each color within = one experiment
# This shows whether experiment A, B, or C wins for each model type
experiments = ["A: Variability Only", "B: BBRef Quality Only", "C: All Features Combined"]
exp_colors  = ["#3498db", "#e74c3c", "#2ecc71"]
models_list = all_results["Model"].unique()
x     = np.arange(len(models_list))
width = 0.25

for i, (exp, color) in enumerate(zip(experiments, exp_colors)):
    subset = all_results[all_results["Experiment"]==exp].set_index("Model")
    vals   = [subset.loc[m,"CV_R2"] if m in subset.index else 0 for m in models_list]
    axes[0].bar(x + i*width, vals, width, label=exp, color=color,
                alpha=0.85, edgecolor="white")

axes[0].axhline(0, color="black", lw=0.8)
axes[0].axhline(0.1, color="grey", lw=0.6, ls=":", alpha=0.7, label="R²=0.1 threshold")
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(models_list, rotation=25, ha="right", fontsize=9)
axes[0].set_ylabel("5-Fold Cross-Validated R²", fontsize=11)
axes[0].set_title("Model Performance by Experiment\n"
                  "(blue=variability only, red=BBRef only, green=combined)",
                  fontsize=11, fontweight="bold")
axes[0].legend(fontsize=8)

# ── Right chart: best model per experiment
# Answers: which single approach is strongest?
best_per_exp = (all_results.loc[all_results.groupby("Experiment")["CV_R2"].idxmax()]
                .sort_values("CV_R2", ascending=False))
bar_colors = exp_colors[::-1][:len(best_per_exp)]

bars = axes[1].bar(range(len(best_per_exp)), best_per_exp["CV_R2"],
                   color=["#2ecc71","#3498db","#e74c3c"][:len(best_per_exp)],
                   edgecolor="white", width=0.5)
axes[1].set_xticks(range(len(best_per_exp)))
axes[1].set_xticklabels(best_per_exp["Experiment"], rotation=12, ha="right", fontsize=9)
axes[1].set_ylabel("Best CV R²", fontsize=11)
axes[1].set_title("Best CV R² per Experiment\n(which feature set is most predictive?)",
                  fontsize=11, fontweight="bold")
for bar, row in zip(bars, best_per_exp.itertuples()):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.003,
                f"{row.CV_R2:.3f}\n({row.Model})",
                ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

print("\nBest model summary per experiment:")
print(best_per_exp[["Experiment","Model","CV_R2","CV_RMSE","Overfit"]].to_string(index=False))
print("\nINTERPRETATION:")
print("If Experiment C (combined) beats A and B, combining quality + variability")
print("features gives more predictive power than either alone.")
print("If B > A, raw talent predicts better than consistency measures.")

## Section 9: Hyperparameter Tuning

We fine-tune the two best ensemble models (Random Forest and Gradient Boosting) using **GridSearchCV** — a systematic search across parameter combinations evaluated by cross-validated R².

**Key parameters:**
- `n_estimators` — how many trees to build (more = better but slower)
- `max_depth` — how deep each tree grows (deeper = more complex, more overfit risk)
- `min_samples_leaf` — minimum data points required in a leaf node (higher = simpler trees)
- `learning_rate` (GB only) — how much each new tree corrects errors (lower = more conservative)

In [ ]:
# ── Tune Random Forest on the combined feature set (Experiment C)
print("Tuning Random Forest...")
rf_grid = {
    "mdl__n_estimators":     [200, 400, 600],    # number of trees
    "mdl__max_depth":        [3, 4, 5, None],    # tree depth (None = unlimited)
    "mdl__min_samples_leaf": [2, 3, 5],          # minimum samples per leaf
}
rf_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                    ("scl", RobustScaler()),
                    ("mdl", RandomForestRegressor(random_state=RANDOM_STATE))])
rf_search = GridSearchCV(rf_pipe, rf_grid, cv=kf, scoring="r2", n_jobs=-1)
rf_search.fit(X_C, y_C)
print(f"  Best params: {rf_search.best_params_}")
print(f"  Best CV R²:  {rf_search.best_score_:.4f}")

# ── Tune Gradient Boosting
print("\nTuning Gradient Boosting...")
gb_grid = {
    "mdl__n_estimators":  [150, 300, 500],
    "mdl__learning_rate": [0.03, 0.05, 0.1],
    "mdl__max_depth":     [2, 3, 4],
}
gb_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                    ("scl", RobustScaler()),
                    ("mdl", GradientBoostingRegressor(subsample=0.8,
                                                      random_state=RANDOM_STATE))])
gb_search = GridSearchCV(gb_pipe, gb_grid, cv=kf, scoring="r2", n_jobs=-1)
gb_search.fit(X_C, y_C)
print(f"  Best params: {gb_search.best_params_}")
print(f"  Best CV R²:  {gb_search.best_score_:.4f}")

# ── Select the best of the two
best_search = rf_search if rf_search.best_score_ >= gb_search.best_score_ else gb_search
best_name   = "Random Forest" if best_search is rf_search else "Gradient Boosting"
print(f"\n✓ Best overall model: {best_name}")
print(f"  Tuned CV R²: {best_search.best_score_:.4f}")
print(f"  Interpretation: This model explains {best_search.best_score_*100:.1f}% of the")
print(f"  variance in playoff win% from our feature set.")

## Section 10: Feature Importance — What Actually Drives Playoff Success?

We use **permutation importance** — the most reliable method for understanding what a model relies on.

**How it works:** For each feature, we randomly shuffle its values across all rows (breaking its relationship with the target) and measure how much the model's R² drops. A big drop = the model depends heavily on that feature. A small drop = the feature is mostly noise.

**Why permutation importance instead of tree-native importance?**
Tree-native importance (based on impurity reduction) tends to favor features with many unique values and can be misleading. Permutation importance is model-agnostic and more trustworthy.

**Feature categories:**
- 🟢 Mean Level — average performance metrics
- 🟣 Floor/Ceiling — p10 and p90 metrics
- 🟠 Volatility — std, CV, IQR, skew metrics
- 🔴 BBRef Quality — VORP, BPM, WS, NetRtg

In [ ]:
# ── Fit the best model on all data for importance computation
best_search.fit(X_C, y_C)

# ── Run permutation importance: shuffle each feature 30 times, measure R² drop
# n_repeats=30 gives stable estimates by averaging over 30 shuffles
perm = permutation_importance(best_search, X_C, y_C,
                               n_repeats=30, random_state=RANDOM_STATE,
                               scoring="r2")

imp_df = pd.DataFrame({
    "Feature":    ALL_FEATURES,
    "Importance": perm.importances_mean,   # avg R² drop across 30 shuffles
    "Std":        perm.importances_std,    # variability of that drop
}).sort_values("Importance", ascending=False).reset_index(drop=True)

# ── Categorize each feature for the summary chart
def categorize(f):
    if f in BBREF_FEATURES:           return "BBRef Quality"
    elif any(k in f for k in ["std","cv","iqr","skew"]): return "Volatility"
    elif any(k in f for k in ["p10","p90"]):             return "Floor/Ceiling"
    else:                                                  return "Mean Level"

imp_df["Category"] = imp_df["Feature"].apply(categorize)
cat_colors = {"BBRef Quality": "#e74c3c", "Volatility": "#e67e22",
              "Floor/Ceiling": "#8e44ad", "Mean Level": "#2ecc71"}
colors = [cat_colors[c] for c in imp_df["Category"]]

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# ── Left: Full feature importance chart
# Error bars show variability across the 30 permutation repeats
# Features near zero had almost no impact when shuffled
axes[0].barh(imp_df["Feature"], imp_df["Importance"],
             xerr=imp_df["Std"], color=colors, edgecolor="white", height=0.7)
axes[0].axvline(0, color="black", lw=0.8)
axes[0].set_xlabel("Mean Decrease in R² when feature is shuffled", fontsize=10)
axes[0].set_title(f"Permutation Feature Importance ({best_name})\n"
                  "Longer bar = model depends more on this feature",
                  fontsize=11, fontweight="bold")
axes[0].invert_yaxis()

from matplotlib.patches import Patch
legend_els = [Patch(facecolor=v, label=k) for k, v in cat_colors.items()]
axes[0].legend(handles=legend_els, loc="lower right", fontsize=9)

# ── Right: Category totals
# Sums up importance by category to answer: what TYPE of feature matters most?
cat_totals = (imp_df.groupby("Category")["Importance"]
              .apply(lambda x: x[x > 0].sum())
              .reset_index()
              .sort_values("Importance", ascending=False))
cat_totals["color"] = cat_totals["Category"].map(cat_colors)

axes[1].bar(cat_totals["Category"], cat_totals["Importance"],
            color=cat_totals["color"], edgecolor="white", width=0.5)
axes[1].set_ylabel("Total Permutation Importance (sum of positive values)", fontsize=10)
axes[1].set_title("Which Category of Feature Matters Most?\n"
                  "This directly answers: consistency vs. quality vs. ceiling",
                  fontsize=11, fontweight="bold")
for i, row in cat_totals.reset_index(drop=True).iterrows():
    axes[1].text(i, row["Importance"]+0.002,
                f"{row['Importance']:.3f}", ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

print("\nTop 10 most important individual features:")
print(imp_df[["Feature","Category","Importance","Std"]].head(10).to_string(index=False))
print("\nCategory totals:")
print(cat_totals[["Category","Importance"]].to_string(index=False))

## Section 11: LassoCV — Which Features Survive Regularization?

LassoCV automatically tunes its regularization strength and drives unimportant feature coefficients to **exactly zero** — a form of automatic feature selection.

**How to read this chart:**
- Only features with non-zero coefficients are shown — Lasso eliminated all others
- **Green bars** = positive coefficient → higher value of this feature predicts higher playoff win%
- **Red bars** = negative coefficient → higher value predicts lower playoff win%
- Bar length = how strongly the feature influences predictions (all features are standardized so lengths are comparable)

**`playoff_games` dominance:** The large green bar for playoff_games reflects that teams that go deeper in the playoffs play more games AND win more — this is partially a mathematical artifact (more wins = more games = higher win%). It's a legitimate predictor but its dominance partly reflects survivorship bias.

In [ ]:
# ── Impute missing values before LassoCV (Lasso can't handle NaN)
imp = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imp.fit_transform(X_C), columns=ALL_FEATURES)

# ── Standardize features so coefficients are comparable across different scales
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

# ── LassoCV: automatically finds optimal alpha (regularization strength) via CV
lasso_cv = LassoCV(cv=kf, max_iter=20000, random_state=RANDOM_STATE)
lasso_cv.fit(X_scaled, y_C)

print(f"Optimal alpha (regularization strength): {lasso_cv.alpha_:.5f}")
print(f"  Lower alpha = less regularization = more features kept")
print(f"  Higher alpha = stronger regularization = more features zeroed out")

# ── Extract non-zero coefficients
coef_df = pd.DataFrame({
    "Feature":     ALL_FEATURES,
    "Coefficient": lasso_cv.coef_,
}).sort_values("Coefficient", key=abs, ascending=False)

nonzero = coef_df[coef_df["Coefficient"].abs() > 1e-6].copy()
nonzero["Category"] = nonzero["Feature"].apply(categorize)

print(f"\nFeatures kept by Lasso: {len(nonzero)} out of {len(ALL_FEATURES)}")
print(f"Features zeroed out:    {len(ALL_FEATURES) - len(nonzero)}")
print("\nSurviving features and their coefficients:")
print(nonzero[["Feature","Category","Coefficient"]].to_string(index=False))

# ── Visualization
fig, ax = plt.subplots(figsize=(11, max(5, len(nonzero)*0.5)))
bar_colors = ["#2ecc71" if v > 0 else "#e74c3c" for v in nonzero["Coefficient"]]
ax.barh(nonzero["Feature"], nonzero["Coefficient"],
        color=bar_colors, edgecolor="white", height=0.65)
ax.axvline(0, color="black", lw=1.0)
ax.set_xlabel("Lasso Coefficient (standardized — all features on same scale)", fontsize=11)
ax.set_title("LassoCV — Features That Survived Automatic Regularization\n"
             "Green = boosts win%  |  Red = hurts win%  |  "
             "All others were zeroed out as uninformative",
             fontsize=11, fontweight="bold")
ax.invert_yaxis()

# Annotate each bar with its category
for i, (_, row) in enumerate(nonzero.iterrows()):
    x_pos = row["Coefficient"] + (0.002 if row["Coefficient"] > 0 else -0.002)
    ha    = "left" if row["Coefficient"] > 0 else "right"
    ax.text(x_pos, i, f"[{row['Category']}]", va="center", ha=ha, fontsize=7, color="grey")

plt.tight_layout()
plt.show()

print("\nINTERPRETATION:")
print("Features Lasso kept are the ones with genuine predictive signal.")
print("The large playoff_games coefficient reflects that winning teams play more games.")
print("Beyond that, PER, WS/48, and pct_elite_TS are the strongest surviving quality/consistency signals.")

## Section 12: Bottom Line Answer

This section synthesizes everything — the correlations, the ML experiments, the feature importance, and the Lasso results — into a direct answer to the research question.

In [ ]:
# ── Aggregate importance by category
cat_signal = (imp_df.groupby("Category")["Importance"]
              .apply(lambda x: x[x > 0].sum())
              .to_dict())
total = sum(cat_signal.values()) + 1e-9

print("=" * 65)
print("BOTTOM LINE: Does Consistency or Volatility Drive Playoff Success?")
print("=" * 65)

print("\n1. FEATURE IMPORTANCE BY CATEGORY (permutation, combined model):")
for cat, val in sorted(cat_signal.items(), key=lambda x: -x[1]):
    print(f"   {cat:<20} {val/total*100:>5.1f}%  (importance: {val:.4f})")

winner = max(cat_signal.items(), key=lambda x: x[1])[0]
print(f"\n   ➤ Dominant driver: {winner}")

print(f"\n2. BEST MODEL PERFORMANCE:")
print(f"   Model:   {best_name}")
print(f"   CV R²:   {best_search.best_score_:.4f}")
print(f"   Meaning: This model explains {best_search.best_score_*100:.1f}% of variance in playoff win%")

print(f"\n3. KEY CORRELATIONS:")
key_feats = [("WS/48","Quality"),("NetRtg","Quality"),("VORP","Quality"),
             ("BPM","Quality"),("cv_OIS_per75","Volatility"),("std_TS_pct","Volatility"),
             ("p90_OIS_per75","Ceiling")]
for feat, cat in key_feats:
    sub = df[[feat,TARGET]].dropna()
    if len(sub) > 10:
        r, p = pearsonr(sub[feat], sub[TARGET])
        sig = " ← significant" if p < 0.05 else ""
        print(f"   {feat:<22} r={r:>6.3f}  p={p:.4f}  [{cat}]{sig}")

print(f"\n4. DIRECT ANSWER TO THE RESEARCH QUESTION:")
print(f"   The variance of a superstar's offensive impact metric has")
print(f"   WEAK predictive power on its own for playoff winning percentage.")
print(f"   Pure volatility measures (std, CV of OIS and TS%) do not")
print(f"   significantly separate playoff winners from losers.")
print(f"")
print(f"   What DOES predict playoff success:")
print(f"   • Sustained overall excellence (WS/48, NetRtg, VORP, BPM)")
print(f"   • The ability to have dominant ceiling games (p90 OIS)")
print(f"   • Shooting efficiency floor — not turning the ball over")
print(f"")
print(f"   CONCLUSION: Raw talent and sustained quality matter far more")
print(f"   than game-to-game consistency. Being a great player is what")
print(f"   drives playoff wins — not how predictable that greatness is.")

## Section 13: Final Summary Dashboard

A clean, publication-ready summary of the three key findings from this project.

In [ ]:
fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)

# ── Panel A: Experiment comparison (which feature set wins?)
ax_a = fig.add_subplot(gs[0, :2])
best_per_exp = (all_results.loc[all_results.groupby("Experiment")["CV_R2"].idxmax()]
                .sort_values("CV_R2", ascending=True))
exp_palette  = {"A: Variability Only":    "#3498db",
                "B: BBRef Quality Only":  "#e74c3c",
                "C: All Features Combined": "#2ecc71"}
bar_cols = [exp_palette.get(e, "#95a5a6") for e in best_per_exp["Experiment"]]

bars = ax_a.barh(
    best_per_exp["Experiment"] + "\n(best: " + best_per_exp["Model"] + ")",
    best_per_exp["CV_R2"],
    color=bar_cols, edgecolor="white", height=0.45
)
ax_a.axvline(0, color="black", lw=0.8)
ax_a.set_xlabel("Best 5-Fold CV R²", fontsize=11)
ax_a.set_title("Which Feature Set Best Predicts Playoff Win %?\n"
               "Higher R² = model explains more variance in outcomes",
               fontsize=11, fontweight="bold")
for bar, row in zip(bars, best_per_exp.itertuples()):
    ax_a.text(max(bar.get_width(), 0) + 0.005,
             bar.get_y() + bar.get_height()/2,
             f"R² = {row.CV_R2:.3f}", va="center", fontsize=10, fontweight="bold")

# ── Panel B: Category importance pie
ax_b = fig.add_subplot(gs[0, 2])
pie_data = [(v, k) for k, v in cat_signal.items() if v > 0]
pie_data.sort(reverse=True)
pie_vals, pie_lbls = zip(*pie_data)
pie_cols = [cat_colors[l] for l in pie_lbls]
wedges, texts, autotexts = ax_b.pie(
    pie_vals, labels=pie_lbls, colors=pie_cols,
    autopct="%1.0f%%", startangle=90,
    textprops={"fontsize": 9}
)
ax_b.set_title("What Type of Feature\nDrives Playoff Win %?\n"
               "(permutation importance)",
               fontsize=11, fontweight="bold")

# ── Panel C: WS/48 vs Playoff Win% (strongest quality predictor)
ax_c = fig.add_subplot(gs[1, 0])
sub = df[["WS/48", TARGET]].dropna()
ax_c.scatter(sub["WS/48"], sub[TARGET], color="#e74c3c", alpha=0.75,
             s=55, edgecolors="white", linewidths=0.4)
m, b, r, p, _ = stats.linregress(sub["WS/48"], sub[TARGET])
x_l = np.linspace(sub["WS/48"].min(), sub["WS/48"].max(), 100)
ax_c.plot(x_l, m*x_l+b, "k--", lw=1.5, label=f"r={r:.2f}*")
ax_c.legend(fontsize=9)
ax_c.set_xlabel("Win Shares per 48 min", fontsize=10)
ax_c.set_ylabel("Playoff Win %", fontsize=10)
ax_c.set_title("Best Quality Predictor:\nWin Shares / 48",
               fontsize=10, fontweight="bold")

# ── Panel D: p90 OIS vs Playoff Win% (strongest variability predictor)
ax_d = fig.add_subplot(gs[1, 1])
sub2 = df[["p90_OIS_per75", TARGET]].dropna()
ax_d.scatter(sub2["p90_OIS_per75"], sub2[TARGET], color="#8e44ad", alpha=0.75,
             s=55, edgecolors="white", linewidths=0.4)
m2, b2, r2v, p2, _ = stats.linregress(sub2["p90_OIS_per75"], sub2[TARGET])
x_l2 = np.linspace(sub2["p90_OIS_per75"].min(), sub2["p90_OIS_per75"].max(), 100)
ax_d.plot(x_l2, m2*x_l2+b2, "k--", lw=1.5, label=f"r={r2v:.2f}*")
ax_d.legend(fontsize=9)
ax_d.set_xlabel("OIS Ceiling (90th percentile per 75)", fontsize=10)
ax_d.set_ylabel("", fontsize=10)
ax_d.set_title("Best Variability Predictor:\nCeiling (p90 of OIS)",
               fontsize=10, fontweight="bold")

# ── Panel E: CV vs Playoff Win% (does pure volatility matter?)
ax_e = fig.add_subplot(gs[1, 2])
sub3 = df[["cv_OIS_per75", TARGET]].dropna()
ax_e.scatter(sub3["cv_OIS_per75"], sub3[TARGET], color="#e67e22", alpha=0.75,
             s=55, edgecolors="white", linewidths=0.4)
m3, b3, r3, p3, _ = stats.linregress(sub3["cv_OIS_per75"], sub3[TARGET])
x_l3 = np.linspace(sub3["cv_OIS_per75"].min(), sub3["cv_OIS_per75"].max(), 100)
sig3 = "*" if p3 < 0.05 else " (not sig.)"
ax_e.plot(x_l3, m3*x_l3+b3, "k--", lw=1.5, label=f"r={r3:.2f}{sig3}")
ax_e.legend(fontsize=9)
ax_e.set_xlabel("CV of OIS per 75 (Volatility)", fontsize=10)
ax_e.set_ylabel("", fontsize=10)
ax_e.set_title("Pure Volatility Predictor:\nCV of OIS (is this significant?)",
               fontsize=10, fontweight="bold")

fig.suptitle(
    "Project 2 Summary: Superstar Offensive Consistency vs. Volatility & Playoff Success\n"
    "Conclusion: Sustained Quality (WS/48, NetRtg) > Ceiling (p90 OIS) > Pure Volatility (CV, Std)",
    fontsize=13, fontweight="bold", y=1.01
)
plt.savefig("project2_final_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nFinal summary chart saved to: project2_final_summary.png")